In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Check device
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: mps


In [2]:
x = torch.randn(3,3).to(device)
print(x.device)

mps:0


In [3]:
df = pd.read_csv('../data/irrigation.csv')

# Define feature groups
NUMERIC_FEATURES = [
    'Soil_Moisture', 'Temperature_C', 'Humidity', 'Rainfall_mm',
    'Soil_pH', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Organic_Carbon',
    'Electrical_Conductivity', 'Previous_Irrigation_mm', 'Field_Area_hectare'
]

CATEGORICAL_FEATURES = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Mulching_Used', 'Region'
]

TARGET = 'Irrigation_Need'
TARGET_MAP = {'Low': 0, 'Medium': 1, 'High': 2}

# Encode categoricals
df_model = df.copy()
le_dict = {}
cat_dims = []  # number of unique values per categorical feature

for col in CATEGORICAL_FEATURES:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le
    cat_dims.append(len(le.classes_))
    print(f"{col}: {len(le.classes_)} unique values")

# Scale numeric features
scaler = StandardScaler()
df_model[NUMERIC_FEATURES] = scaler.fit_transform(df_model[NUMERIC_FEATURES])

# Encode target
df_model[TARGET] = df_model[TARGET].map(TARGET_MAP)

print(f"\nCategorical dims: {cat_dims}")
print(f"Numeric features: {len(NUMERIC_FEATURES)}")

Soil_Type: 4 unique values
Crop_Type: 6 unique values
Crop_Growth_Stage: 4 unique values
Season: 3 unique values
Mulching_Used: 2 unique values
Region: 5 unique values

Categorical dims: [4, 6, 4, 3, 2, 5]
Numeric features: 11


In [4]:
from sklearn.model_selection import train_test_split

X_cat = df_model[CATEGORICAL_FEATURES].values
X_num = df_model[NUMERIC_FEATURES].values
y     = df_model[TARGET].values

# Stratified split
X_cat_train, X_cat_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_cat, X_num, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

class IrrigationDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.y     = torch.tensor(y,     dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_cat[idx], self.X_num[idx], self.y[idx]

train_dataset = IrrigationDataset(X_cat_train, X_num_train, y_train)
test_dataset  = IrrigationDataset(X_cat_test,  X_num_test,  y_test)

train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 125
Test batches:  32


In [5]:
class TabTransformer(nn.Module):
    def __init__(
        self,
        cat_dims,           # list of unique counts per categorical feature
        num_continuous,     # number of numeric features
        embed_dim=32,       # embedding size for each categorical
        num_heads=8,        # attention heads
        num_layers=3,       # transformer layers
        ff_dim=128,         # feedforward hidden dim
        dropout=0.1,
        num_classes=3
    ):
        super().__init__()

        # One embedding per categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(dim + 1, embed_dim) for dim in cat_dims
        ])

        # Transformer encoder for categorical embeddings
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # MLP head — combines transformer output + numeric features
        cat_output_dim = len(cat_dims) * embed_dim
        total_dim      = cat_output_dim + num_continuous

        self.mlp = nn.Sequential(
            nn.Linear(total_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x_cat, x_num):
        # Embed each categorical feature → shape: (batch, num_cats, embed_dim)
        cat_embeds = torch.stack(
            [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)],
            dim=1
        )

        # Pass through transformer
        transformed = self.transformer(cat_embeds)

        # Flatten transformer output
        transformed = transformed.flatten(start_dim=1)

        # Concatenate with numeric features
        combined = torch.cat([transformed, x_num], dim=1)

        # Final classification
        return self.mlp(combined)


# Instantiate model
model = TabTransformer(
    cat_dims=cat_dims,
    num_continuous=len(NUMERIC_FEATURES),
    embed_dim=32,
    num_heads=8,
    num_layers=3,
    ff_dim=128,
    dropout=0.1,
    num_classes=3
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

TabTransformer(
  (embeddings): ModuleList(
    (0): Embedding(5, 32)
    (1): Embedding(7, 32)
    (2): Embedding(5, 32)
    (3): Embedding(4, 32)
    (4): Embedding(3, 32)
    (5): Embedding(6, 32)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (mlp): Sequential(
    (0): Linear(in_features=203, out_features=256, bias=True)
    (1): R

Training

In [6]:
# Class weights to handle imbalance
class_counts  = np.bincount(y_train)
class_weights = torch.tensor(
    1.0 / class_counts, dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

EPOCHS = 30

train_losses = []
train_accs   = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for x_cat, x_num, labels in train_loader:
        x_cat  = x_cat.to(device)
        x_num  = x_num.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(x_cat, x_num)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    scheduler.step()

    avg_loss = total_loss / len(train_loader)
    acc      = accuracy_score(all_labels, all_preds)
    train_losses.append(avg_loss)
    train_accs.append(acc)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc: {acc*100:.2f}%")

Epoch 05/30 | Loss: 0.4307 | Train Acc: 78.74%
Epoch 10/30 | Loss: 0.3132 | Train Acc: 84.17%
Epoch 15/30 | Loss: 0.2189 | Train Acc: 89.28%
Epoch 20/30 | Loss: 0.1643 | Train Acc: 91.50%
Epoch 25/30 | Loss: 0.1284 | Train Acc: 93.39%
Epoch 30/30 | Loss: 0.1137 | Train Acc: 94.25%


Evaluation

In [7]:
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for x_cat, x_num, labels in test_loader:
        x_cat   = x_cat.to(device)
        x_num   = x_num.to(device)
        outputs = model(x_cat, x_num)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='macro')

print("=== Tab Transformer Results ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Macro F1 : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Low','Medium','High']))

=== Tab Transformer Results ===
Accuracy : 90.55%
Macro F1 : 0.8248

Classification Report:
              precision    recall  f1-score   support

         Low       0.96      0.92      0.94      1173
      Medium       0.86      0.90      0.88       760
        High       0.64      0.67      0.66        67

    accuracy                           0.91      2000
   macro avg       0.82      0.83      0.82      2000
weighted avg       0.91      0.91      0.91      2000



In [8]:
results = {
    'Model'       : ['Decision Tree', 'Random Forest', 'Tab Transformer'],
    'Accuracy'    : [99.25,           98.45,           round(acc*100, 2)],
    'Macro F1'    : [0.97,            0.90,            round(f1, 4)],
    'High F1'     : [0.92,            0.74,            '(from report above)']
}

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

          Model  Accuracy  Macro F1             High F1
  Decision Tree     99.25    0.9700                0.92
  Random Forest     98.45    0.9000                0.74
Tab Transformer     90.55    0.8248 (from report above)


In [10]:
import os
os.makedirs('../models', exist_ok=True)

torch.save(model.state_dict(), '../models/tab_transformer.pt')
import pickle
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print(" Tab Transformer saved → models/tab_transformer.pt")
print(" Scaler saved         → models/scaler.pkl")

 Tab Transformer saved → models/tab_transformer.pt
 Scaler saved         → models/scaler.pkl


Extended Training (50 more epochs)

In [11]:
EPOCHS_2 = 50

for epoch in range(EPOCHS_2):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for x_cat, x_num, labels in train_loader:
        x_cat  = x_cat.to(device)
        x_num  = x_num.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(x_cat, x_num)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(train_loader)
    acc_train = accuracy_score(all_labels, all_preds)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS_2} | Loss: {avg_loss:.4f} | Train Acc: {acc_train*100:.2f}%")

Epoch 10/50 | Loss: 0.0873 | Train Acc: 95.50%
Epoch 20/50 | Loss: 0.0767 | Train Acc: 96.25%
Epoch 30/50 | Loss: 0.0580 | Train Acc: 97.34%
Epoch 40/50 | Loss: 0.0530 | Train Acc: 97.32%
Epoch 50/50 | Loss: 0.0371 | Train Acc: 98.31%


In [12]:
model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for x_cat, x_num, labels in test_loader:
        x_cat   = x_cat.to(device)
        x_num   = x_num.to(device)
        outputs = model(x_cat, x_num)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, average='macro')

print("=== Tab Transformer Results (80 epochs total) ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Macro F1 : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=['Low','Medium','High']))

=== Tab Transformer Results (80 epochs total) ===
Accuracy : 91.20%
Macro F1 : 0.8279

Classification Report:
              precision    recall  f1-score   support

         Low       0.94      0.94      0.94      1173
      Medium       0.88      0.89      0.89       760
        High       0.73      0.60      0.66        67

    accuracy                           0.91      2000
   macro avg       0.85      0.81      0.83      2000
weighted avg       0.91      0.91      0.91      2000



Retraining as overfit occured

In [14]:
# ── Reset model with better hyperparameters ──
model = TabTransformer(
    cat_dims=cat_dims,
    num_continuous=len(NUMERIC_FEATURES),
    embed_dim=32,
    num_heads=8,
    num_layers=3,
    ff_dim=128,
    dropout=0.2,        # increased from 0.1 — reduces overfitting
    num_classes=3
).to(device)

# Lower learning rate + stronger weight decay
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)

# Reduce LR when progress stalls instead of fixed schedule
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

# Recompute class weights
class_counts  = np.bincount(y_train)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 60
best_val_acc = 0
best_preds   = None

print("Retraining with improved settings...\n")

for epoch in range(EPOCHS):
    # ── Train ──
    model.train()
    total_loss = 0
    train_preds, train_labels = [], []

    for x_cat, x_num, labels in train_loader:
        x_cat, x_num, labels = x_cat.to(device), x_num.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(x_cat, x_num)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        train_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

    # ── Validate ──
    model.eval()
    val_preds, val_labels = [], []
    val_loss = 0

    with torch.no_grad():
        for x_cat, x_num, labels in test_loader:
            x_cat, x_num, labels = x_cat.to(device), x_num.to(device), labels.to(device)
            outputs  = model(x_cat, x_num)
            val_loss += criterion(outputs, labels).item()
            val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    avg_train_loss = total_loss / len(train_loader)
    avg_val_loss   = val_loss   / len(test_loader)
    train_acc      = accuracy_score(train_labels, train_preds)
    val_acc        = accuracy_score(val_labels,   val_preds)

    scheduler.step(avg_val_loss)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_preds   = val_preds[:]
        torch.save(model.state_dict(), '../models/tab_transformer_best.pt')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
              f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
              f"Val Acc: {val_acc*100:.2f}% | Best: {best_val_acc*100:.2f}%")

print(f"\nBest validation accuracy: {best_val_acc*100:.2f}%")

Retraining with improved settings...

Epoch 10/60 | Train Loss: 0.4025 | Train Acc: 80.12% | Val Acc: 80.95% | Best: 80.95%
Epoch 20/60 | Train Loss: 0.2776 | Train Acc: 86.12% | Val Acc: 85.60% | Best: 85.60%
Epoch 30/60 | Train Loss: 0.2129 | Train Acc: 89.31% | Val Acc: 86.55% | Best: 86.90%
Epoch 40/60 | Train Loss: 0.2088 | Train Acc: 89.15% | Val Acc: 87.00% | Best: 87.50%
Epoch 50/60 | Train Loss: 0.1974 | Train Acc: 89.65% | Val Acc: 86.95% | Best: 87.50%
Epoch 60/60 | Train Loss: 0.1925 | Train Acc: 90.01% | Val Acc: 87.15% | Best: 87.50%

Best validation accuracy: 87.50%


In [15]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

acc = accuracy_score(val_labels, best_preds)
f1  = f1_score(val_labels, best_preds, average='macro')

print("=== Tab Transformer Best Results ===")
print(f"Accuracy : {acc*100:.2f}%")
print(f"Macro F1 : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(val_labels, best_preds, target_names=['Low','Medium','High']))

=== Tab Transformer Best Results ===
Accuracy : 87.50%
Macro F1 : 0.7875

Classification Report:
              precision    recall  f1-score   support

         Low       0.92      0.92      0.92      1173
      Medium       0.85      0.82      0.83       760
        High       0.54      0.70      0.61        67

    accuracy                           0.88      2000
   macro avg       0.77      0.81      0.79      2000
weighted avg       0.88      0.88      0.88      2000

